# Ronaldo Appearance Probability Model

This notebook builds the model that estimates the probability that Cristiano Ronaldo
appears in an eligible official fixture.

Target:

P(appearance | eligible fixture)

The model is trained on the harmonized appearance-opportunity dataset containing
1,584 eligible fixtures: 1,333 appearances and 251 eligible nonappearances.

All engineered predictors must be available before kickoff to prevent future leakage.

In [3]:
from pathlib import Path

import pandas as pd


# Paths
appearance_path = Path(
    "../data/appearance/ronaldo_appearance_opportunities.csv"
)

df = pd.read_csv(
    appearance_path,
    parse_dates=["date"],
)

df = (
    df.sort_values(["date", "team", "opponent"])
    .reset_index(drop=True)
)


print("APPEARANCE FEATURE MODEL — INPUT CHECK")
print("=" * 80)

print("Shape:", df.shape)

print(
    "Date range:",
    df["date"].min().date(),
    "→",
    df["date"].max().date(),
)

print("Appearances:", int(df["appeared"].sum()))

print(
    "Nonappearances:",
    int(df["appeared"].eq(0).sum()),
)

print(
    "Appearance rate:",
    round(df["appeared"].mean(), 4),
)

print("\nColumns:")
print(df.columns.tolist())


# Integrity checks
assert df.shape == (1584, 9)
assert df["appeared"].sum() == 1333
assert df["appeared"].eq(0).sum() == 251

assert df["date"].notna().all()
assert df["appeared"].isin([0, 1]).all()
assert df["date"].is_monotonic_increasing

print("\n✓ Appearance dataset loaded successfully.")
print("✓ Target is binary and valid.")
print("✓ Dataset is chronologically ordered.")
print("✓ Ready for pre-match feature engineering.")

APPEARANCE FEATURE MODEL — INPUT CHECK
Shape: (1584, 9)
Date range: 2002-08-14 → 2026-08-28
Appearances: 1333
Nonappearances: 251
Appearance rate: 0.8415

Columns:
['date', 'team', 'opponent', 'competition', 'venue', 'appeared', 'canonical_match_id', 'season', 'tenure']

✓ Appearance dataset loaded successfully.
✓ Target is binary and valid.
✓ Dataset is chronologically ordered.
✓ Ready for pre-match feature engineering.


In [5]:
# Recent pre-match appearance history

import numpy as np


# Previous eligible fixture appearance
df["appeared_previous_pre"] = (
    df["appeared"]
    .shift(1)
)


# Rolling appearance rates using only prior fixtures
prior_appeared = df["appeared"].shift(1)

df["appearance_rate_last_5_pre"] = (
    prior_appeared
    .rolling(window=5, min_periods=1)
    .mean()
)

df["appearance_rate_last_10_pre"] = (
    prior_appeared
    .rolling(window=10, min_periods=1)
    .mean()
)


# Consecutive appearances immediately before each fixture
streak_pre = []
streak = 0

for appeared in df["appeared"]:

    streak_pre.append(streak)

    if appeared == 1:
        streak += 1
    else:
        streak = 0

df["consecutive_appearances_pre"] = streak_pre


# First career opportunity has no prior fixture
df["appeared_previous_pre"] = (
    df["appeared_previous_pre"]
    .fillna(0)
)

df["appearance_rate_last_5_pre"] = (
    df["appearance_rate_last_5_pre"]
    .fillna(0)
)

df["appearance_rate_last_10_pre"] = (
    df["appearance_rate_last_10_pre"]
    .fillna(0)
)


feature_cols = [
    "appeared_previous_pre",
    "appearance_rate_last_5_pre",
    "appearance_rate_last_10_pre",
    "consecutive_appearances_pre",
]


print("RECENT APPEARANCE FEATURES")
print("=" * 90)

print(
    df[
        [
            "date",
            "team",
            "opponent",
            "appeared",
            *feature_cols,
        ]
    ]
    .head(15)
    .to_string(index=False)
)


print("\nFeature ranges:")

for col in feature_cols:
    print(
        f"{col}: "
        f"{df[col].min():.3f} → "
        f"{df[col].max():.3f}"
    )


print("\nMissing values:")
print(
    df[feature_cols]
    .isna()
    .sum()
    .to_string()
)


# Leakage checks
assert df[feature_cols].isna().sum().sum() == 0

assert (
    df["appeared_previous_pre"].iloc[1:].to_numpy()
    ==
    df["appeared"].iloc[:-1].to_numpy()
).all()

assert df["appearance_rate_last_5_pre"].between(0, 1).all()
assert df["appearance_rate_last_10_pre"].between(0, 1).all()

assert (df["consecutive_appearances_pre"] >= 0).all()


print("\n✓ Recent appearance features created.")
print("✓ Every rolling feature uses only previous fixtures.")
print("✓ Current-fixture target is excluded from its predictors.")

RECENT APPEARANCE FEATURES
      date        team          opponent  appeared  appeared_previous_pre  appearance_rate_last_5_pre  appearance_rate_last_10_pre  consecutive_appearances_pre
2002-08-14 Sporting CP       Inter Milan         1                    0.0                    0.000000                     0.000000                            0
2002-08-18 Sporting CP           Leixões         0                    1.0                    1.000000                     1.000000                            1
2002-08-23 Sporting CP         Académica         0                    0.0                    0.500000                     0.500000                            0
2002-08-27 Sporting CP       Inter Milan         0                    0.0                    0.333333                     0.333333                            0
2002-09-01 Sporting CP       Santa Clara         0                    0.0                    0.250000                     0.250000                            0
2002-09-14 Sp

In [7]:
# Pre-match schedule and workload context

dates = df["date"]


# Days since Ronaldo's previous eligible fixture opportunity
df["days_since_last_fixture_pre"] = (
    dates.diff().dt.days
)


# Number of previous eligible fixtures in the last N days
fixtures_7 = []
fixtures_14 = []

for i, current_date in enumerate(dates):

    previous_dates = dates.iloc[:i]

    fixtures_7.append(
        int(
            (
                (previous_dates >= current_date - pd.Timedelta(days=7))
                & (previous_dates < current_date)
            ).sum()
        )
    )

    fixtures_14.append(
        int(
            (
                (previous_dates >= current_date - pd.Timedelta(days=14))
                & (previous_dates < current_date)
            ).sum()
        )
    )


df["fixtures_last_7_days_pre"] = fixtures_7
df["fixtures_last_14_days_pre"] = fixtures_14


# No previous opportunity exists for the first row
df["days_since_last_fixture_pre"] = (
    df["days_since_last_fixture_pre"]
    .fillna(0)
)


schedule_features = [
    "days_since_last_fixture_pre",
    "fixtures_last_7_days_pre",
    "fixtures_last_14_days_pre",
]


print("PRE-MATCH SCHEDULE / WORKLOAD FEATURES")
print("=" * 95)

print(
    df[
        [
            "date",
            "team",
            "opponent",
            "appeared",
            *schedule_features,
        ]
    ]
    .head(20)
    .to_string(index=False)
)


print("\nFeature ranges:")

for col in schedule_features:
    print(
        f"{col}: "
        f"{df[col].min()} → {df[col].max()}"
    )


print("\nMissing values:")
print(
    df[schedule_features]
    .isna()
    .sum()
    .to_string()
)


assert df[schedule_features].isna().sum().sum() == 0

assert (df["days_since_last_fixture_pre"] >= 0).all()
assert (df["fixtures_last_7_days_pre"] >= 0).all()
assert (df["fixtures_last_14_days_pre"] >= 0).all()

assert (
    df["fixtures_last_14_days_pre"]
    >= df["fixtures_last_7_days_pre"]
).all()


print("\n✓ Schedule/workload features created.")
print("✓ Only fixtures occurring before the current fixture are counted.")
print("✓ No current-fixture target information is used.")

PRE-MATCH SCHEDULE / WORKLOAD FEATURES
      date        team            opponent  appeared  days_since_last_fixture_pre  fixtures_last_7_days_pre  fixtures_last_14_days_pre
2002-08-14 Sporting CP         Inter Milan         1                          0.0                         0                          0
2002-08-18 Sporting CP             Leixões         0                          4.0                         1                          1
2002-08-23 Sporting CP           Académica         0                          5.0                         1                          2
2002-08-27 Sporting CP         Inter Milan         0                          4.0                         1                          3
2002-09-01 Sporting CP         Santa Clara         0                          5.0                         1                          3
2002-09-14 Sporting CP      Paços Ferreira         0                         13.0                         0                          1
2002-09-19 Sport

In [9]:
# Pre-match age and fixture context

ronaldo_birth_date = pd.Timestamp("1985-02-05")


# Exact age at the fixture
df["age_years_pre"] = (
    (df["date"] - ronaldo_birth_date).dt.days
    / 365.2425
)


# National-team fixture
df["is_national_team"] = (
    df["team"].eq("Portugal")
    .astype(int)
)


# Basic calendar context
df["fixture_month"] = df["date"].dt.month
df["fixture_year"] = df["date"].dt.year


context_features = [
    "age_years_pre",
    "is_national_team",
    "fixture_month",
    "fixture_year",
]


print("AGE / FIXTURE CONTEXT FEATURES")
print("=" * 90)

print(
    df[
        [
            "date",
            "team",
            "competition",
            "appeared",
            *context_features,
        ]
    ]
    .iloc[[0, 100, 500, 1000, -1]]
    .to_string(index=False)
)


print("\nFeature ranges:")

for col in context_features:
    print(
        f"{col}: "
        f"{df[col].min():.3f} → "
        f"{df[col].max():.3f}"
    )


print("\nNational-team opportunities:")
print(
    df["is_national_team"]
    .value_counts()
    .sort_index()
    .to_string()
)


print("\nMissing values:")
print(
    df[context_features]
    .isna()
    .sum()
    .to_string()
)


assert df[context_features].isna().sum().sum() == 0

assert df["age_years_pre"].between(17, 42).all()
assert df["is_national_team"].isin([0, 1]).all()
assert df["fixture_month"].between(1, 12).all()

assert df["is_national_team"].sum() == 290


print("\n✓ Age and fixture-context features created.")
print("✓ Portugal opportunities identified correctly.")
print("✓ All features are available before kickoff.")

AGE / FIXTURE CONTEXT FEATURES
      date              team                 competition  appeared  age_years_pre  is_national_team  fixture_month  fixture_year
2002-08-14       Sporting CP Champions League Qualifying         1      17.519867                 0              8          2002
2004-04-24 Manchester United              Premier League         1      19.214631                 0              4          2004
2010-02-13       Real Madrid                     La Liga         1      25.021732                 0              2          2010
2017-04-29       Real Madrid                     La Liga         1      32.227903                 0              4          2017
2026-08-28          Al-Nassr                  Pro League         1      41.558690                 0              8          2026

Feature ranges:
age_years_pre: 17.520 → 41.559
is_national_team: 0.000 → 1.000
fixture_month: 1.000 → 12.000
fixture_year: 2002.000 → 2026.000

National-team opportunities:
is_national_team
0   

In [11]:
# Team-specific pre-match appearance history

df["team_appeared_previous_pre"] = (
    df.groupby("team")["appeared"]
    .shift(1)
)

df["team_appearance_rate_last_5_pre"] = (
    df.groupby("team")["appeared"]
    .transform(
        lambda s: (
            s.shift(1)
            .rolling(window=5, min_periods=1)
            .mean()
        )
    )
)


# No prior history for a new team
df["team_appeared_previous_pre"] = (
    df["team_appeared_previous_pre"]
    .fillna(0)
)

df["team_appearance_rate_last_5_pre"] = (
    df["team_appearance_rate_last_5_pre"]
    .fillna(0)
)


team_history_features = [
    "team_appeared_previous_pre",
    "team_appearance_rate_last_5_pre",
]


print("TEAM-SPECIFIC APPEARANCE HISTORY")
print("=" * 100)

# Show the transition into Portugal to verify that
# Portugal history is kept separate from club history.
sample = df.loc[
    (df["date"] >= "2003-08-01")
    & (df["date"] <= "2003-10-31"),
    [
        "date",
        "team",
        "opponent",
        "appeared",
        "appeared_previous_pre",
        "team_appeared_previous_pre",
        "appearance_rate_last_5_pre",
        "team_appearance_rate_last_5_pre",
    ],
]

print(sample.to_string(index=False))


print("\nFeature ranges:")

for col in team_history_features:
    print(
        f"{col}: "
        f"{df[col].min():.3f} → "
        f"{df[col].max():.3f}"
    )


print("\nMissing values:")
print(
    df[team_history_features]
    .isna()
    .sum()
    .to_string()
)


assert df[team_history_features].isna().sum().sum() == 0

assert df["team_appeared_previous_pre"].isin([0, 1]).all()

assert (
    df["team_appearance_rate_last_5_pre"]
    .between(0, 1)
    .all()
)


print("\n✓ Team-specific appearance history created.")
print("✓ Club and Portugal selection histories remain separate.")
print("✓ Current-fixture target is excluded.")

TEAM-SPECIFIC APPEARANCE HISTORY
      date              team                opponent  appeared  appeared_previous_pre  team_appeared_previous_pre  appearance_rate_last_5_pre  team_appearance_rate_last_5_pre
2003-08-16 Manchester United        Bolton Wanderers         1                    1.0                         0.0                         0.8                         0.000000
2003-08-20          Portugal              Kazakhstan         1                    1.0                         0.0                         1.0                         0.000000
2003-08-23 Manchester United        Newcastle United         1                    1.0                         1.0                         1.0                         1.000000
2003-08-27 Manchester United Wolverhampton Wanderers         1                    1.0                         1.0                         1.0                         1.000000
2003-08-31 Manchester United             Southampton         1                    1.0       

In [13]:
# Step 11.6 — Define the appearance-model feature set

numeric_features = [
    # Recent overall appearance history
    "appeared_previous_pre",
    "appearance_rate_last_5_pre",
    "appearance_rate_last_10_pre",
    "consecutive_appearances_pre",

    # Schedule / workload
    "days_since_last_fixture_pre",
    "fixtures_last_7_days_pre",
    "fixtures_last_14_days_pre",

    # Career / fixture context
    "age_years_pre",
    "is_national_team",
    "fixture_month",

    # Team-specific selection history
    "team_appeared_previous_pre",
    "team_appearance_rate_last_5_pre",
]

categorical_features = [
    "team",
    "competition",
    "venue",
]

target = "appeared"

appearance_features = (
    numeric_features
    + categorical_features
)


print("APPEARANCE MODEL — FEATURE SPECIFICATION")
print("=" * 90)

print("Numeric features:", len(numeric_features))
for feature in numeric_features:
    print(" ", feature)

print("\nCategorical features:", len(categorical_features))
for feature in categorical_features:
    print(" ", feature)

print("\nTotal predictors:", len(appearance_features))
print("Target:", target)


# Basic integrity
missing_features = [
    col
    for col in appearance_features
    if col not in df.columns
]

assert not missing_features, (
    f"Missing features: {missing_features}"
)

assert target not in appearance_features
assert "canonical_match_id" not in appearance_features
assert "season" not in appearance_features
assert "tenure" not in appearance_features
assert "opponent" not in appearance_features
assert "fixture_year" not in appearance_features

assert (
    df[appearance_features]
    .isna()
    .sum()
    .sum()
    == 0
)


print("\nTarget distribution:")
print(
    df[target]
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    "\nPositive rate:",
    round(df[target].mean(), 4),
)


print("\nCategorical cardinality:")

for col in categorical_features:
    print(
        f"{col}: "
        f"{df[col].nunique()} unique values"
    )


print("\n✓ Appearance feature specification frozen.")
print("✓ No target, ID, season or tenure leakage included.")
print("✓ All predictors are available before kickoff.")

APPEARANCE MODEL — FEATURE SPECIFICATION
Numeric features: 12
  appeared_previous_pre
  appearance_rate_last_5_pre
  appearance_rate_last_10_pre
  consecutive_appearances_pre
  days_since_last_fixture_pre
  fixtures_last_7_days_pre
  fixtures_last_14_days_pre
  age_years_pre
  is_national_team
  fixture_month
  team_appeared_previous_pre
  team_appearance_rate_last_5_pre

Categorical features: 3
  team
  competition
  venue

Total predictors: 15
Target: appeared

Target distribution:
appeared
0     251
1    1333

Positive rate: 0.8415

Categorical cardinality:
team: 6 unique values
competition: 33 unique values
venue: 3 unique values

✓ Appearance feature specification frozen.
✓ No target, ID, season or tenure leakage included.
✓ All predictors are available before kickoff.


## Establish development/test boundary

In [16]:
# Establish temporal development/test boundary

TEST_START = pd.Timestamp("2025-01-01")

dev_df = (
    df.loc[df["date"] < TEST_START]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df.loc[df["date"] >= TEST_START]
    .copy()
    .reset_index(drop=True)
)


def split_summary(name, data):
    opportunities = len(data)
    appearances = int(data["appeared"].sum())
    nonappearances = int(data["appeared"].eq(0).sum())

    return {
        "split": name,
        "start": data["date"].min().date(),
        "end": data["date"].max().date(),
        "opportunities": opportunities,
        "appearances": appearances,
        "nonappearances": nonappearances,
        "appearance_rate": appearances / opportunities,
    }


split_audit = pd.DataFrame([
    split_summary("Development", dev_df),
    split_summary("Final Test", test_df),
])


print("APPEARANCE MODEL — TEMPORAL HOLDOUT")
print("=" * 100)

print(
    split_audit.to_string(
        index=False
    )
)


print("\nFinal-test breakdown by team:")

test_team_summary = (
    test_df
    .groupby("team")
    .agg(
        opportunities=("appeared", "size"),
        appearances=("appeared", "sum"),
    )
)

test_team_summary["nonappearances"] = (
    test_team_summary["opportunities"]
    - test_team_summary["appearances"]
)

test_team_summary["appearance_rate"] = (
    test_team_summary["appearances"]
    / test_team_summary["opportunities"]
)

print(
    test_team_summary.to_string()
)


# Integrity checks
assert len(dev_df) + len(test_df) == len(df)

assert dev_df["date"].max() < TEST_START
assert test_df["date"].min() >= TEST_START

assert (
    dev_df["appeared"].sum()
    + test_df["appeared"].sum()
    == 1333
)

assert (
    dev_df["appeared"].eq(0).sum()
    + test_df["appeared"].eq(0).sum()
    == 251
)

assert set(dev_df.index).isdisjoint(
    set(range(len(dev_df), len(dev_df) + len(test_df)))
) or True


print(
    "\n✓ Development and final-test periods separated chronologically."
)

print(
    "✓ 2025+ fixtures are reserved for final evaluation."
)

print(
    "✓ No model fitting or tuning has touched the final test set."
)

APPEARANCE MODEL — TEMPORAL HOLDOUT
      split      start        end  opportunities  appearances  nonappearances  appearance_rate
Development 2002-08-14 2024-12-06           1485         1255             230         0.845118
 Final Test 2025-01-09 2026-08-28             99           78              21         0.787879

Final-test breakdown by team:
          opportunities  appearances  nonappearances  appearance_rate
team                                                                 
Al-Nassr             81           62              19         0.765432
Portugal             18           16               2         0.888889

✓ Development and final-test periods separated chronologically.
✓ 2025+ fixtures are reserved for final evaluation.
✓ No model fitting or tuning has touched the final test set.


In [18]:
# Inspect development-period appearance distribution by year

dev_yearly = (
    dev_df
    .assign(year=dev_df["date"].dt.year)
    .groupby("year")
    .agg(
        opportunities=("appeared", "size"),
        appearances=("appeared", "sum"),
    )
)

dev_yearly["nonappearances"] = (
    dev_yearly["opportunities"]
    - dev_yearly["appearances"]
)

dev_yearly["appearance_rate"] = (
    dev_yearly["appearances"]
    / dev_yearly["opportunities"]
)


print("DEVELOPMENT PERIOD — YEARLY TARGET DISTRIBUTION")
print("=" * 85)

print(
    dev_yearly.to_string(
        formatters={
            "appearance_rate": "{:.4f}".format
        }
    )
)


print("\nTotals")
print("-" * 40)

print(
    "Opportunities:",
    int(dev_yearly["opportunities"].sum()),
)

print(
    "Appearances:",
    int(dev_yearly["appearances"].sum()),
)

print(
    "Nonappearances:",
    int(dev_yearly["nonappearances"].sum()),
)


# Every year used for validation should contain both classes
years_with_no_nonapps = (
    dev_yearly.index[
        dev_yearly["nonappearances"].eq(0)
    ]
    .tolist()
)

print(
    "\nYears with zero nonappearances:",
    years_with_no_nonapps,
)


assert dev_yearly["opportunities"].sum() == 1485
assert dev_yearly["appearances"].sum() == 1255
assert dev_yearly["nonappearances"].sum() == 230

print("\n✓ Development-period distribution verified.")

DEVELOPMENT PERIOD — YEARLY TARGET DISTRIBUTION
      opportunities  appearances  nonappearances appearance_rate
year                                                            
2002             22           16               6          0.7273
2003             54           36              18          0.6667
2004             75           60              15          0.8000
2005             72           64               8          0.8889
2006             66           60               6          0.9091
2007             66           60               6          0.9091
2008             72           58              14          0.8056
2009             69           49              20          0.7101
2010             61           59               2          0.9672
2011             70           60              10          0.8571
2012             74           71               3          0.9595
2013             65           59               6          0.9077
2014             72           60          

In [20]:
# Define expanding temporal validation folds

fold_specs = [
    {
        "fold": 1,
        "val_start": "2017-01-01",
        "val_end": "2018-12-31",
    },
    {
        "fold": 2,
        "val_start": "2019-01-01",
        "val_end": "2020-12-31",
    },
    {
        "fold": 3,
        "val_start": "2021-01-01",
        "val_end": "2022-12-31",
    },
    {
        "fold": 4,
        "val_start": "2023-01-01",
        "val_end": "2024-12-31",
    },
]


temporal_folds = []
fold_audit = []

for spec in fold_specs:

    val_start = pd.Timestamp(spec["val_start"])
    val_end = pd.Timestamp(spec["val_end"])

    train_idx = dev_df.index[
        dev_df["date"] < val_start
    ].tolist()

    val_idx = dev_df.index[
        (dev_df["date"] >= val_start)
        & (dev_df["date"] <= val_end)
    ].tolist()

    train = dev_df.loc[train_idx]
    val = dev_df.loc[val_idx]

    assert len(train) > 0
    assert len(val) > 0

    assert train["date"].max() < val["date"].min()

    # Both classes must exist in every validation fold
    assert val["appeared"].nunique() == 2

    temporal_folds.append({
        "fold": spec["fold"],
        "train_idx": train_idx,
        "val_idx": val_idx,
    })

    fold_audit.append({
        "fold": spec["fold"],

        "train_start":
            train["date"].min().date(),

        "train_end":
            train["date"].max().date(),

        "train_n":
            len(train),

        "train_nonapps":
            int(train["appeared"].eq(0).sum()),

        "val_start":
            val["date"].min().date(),

        "val_end":
            val["date"].max().date(),

        "val_n":
            len(val),

        "val_apps":
            int(val["appeared"].sum()),

        "val_nonapps":
            int(val["appeared"].eq(0).sum()),

        "val_rate":
            val["appeared"].mean(),
    })


fold_audit = pd.DataFrame(fold_audit)


print("APPEARANCE MODEL — EXPANDING TEMPORAL FOLDS")
print("=" * 125)

print(
    fold_audit.to_string(
        index=False,
        formatters={
            "val_rate": "{:.4f}".format
        }
    )
)


print("\nValidation coverage")

all_val_idx = [
    idx
    for fold in temporal_folds
    for idx in fold["val_idx"]
]

print(
    "Total validation predictions:",
    len(all_val_idx),
)

print(
    "Unique validation rows:",
    len(set(all_val_idx)),
)

print(
    "Validation appearances:",
    int(
        dev_df.loc[
            all_val_idx,
            "appeared"
        ].sum()
    ),
)

print(
    "Validation nonappearances:",
    int(
        dev_df.loc[
            all_val_idx,
            "appeared"
        ].eq(0)
        .sum()
    ),
)


assert len(all_val_idx) == len(set(all_val_idx))

assert (
    max(
        dev_df.loc[
            temporal_folds[-1]["val_idx"],
            "date"
        ]
    )
    < TEST_START
)

print("\n✓ Four expanding temporal folds created.")
print("✓ Every validation fold contains both target classes.")
print("✓ Validation periods do not overlap.")
print("✓ Final 2025+ test period remains untouched.")

APPEARANCE MODEL — EXPANDING TEMPORAL FOLDS
 fold train_start  train_end  train_n  train_nonapps  val_start    val_end  val_n  val_apps  val_nonapps val_rate
    1  2002-08-14 2016-12-18      972            146 2017-01-04 2018-12-29    145       113           32   0.7793
    2  2002-08-14 2018-12-29     1117            178 2019-01-12 2020-12-22    115        95           20   0.8261
    3  2002-08-14 2020-12-22     1232            198 2021-01-03 2022-12-10    130       111           19   0.8538
    4  2002-08-14 2022-12-10     1362            217 2023-01-22 2024-12-06    123       110           13   0.8943

Validation coverage
Total validation predictions: 513
Unique validation rows: 513
Validation appearances: 429
Validation nonappearances: 84

✓ Four expanding temporal folds created.
✓ Every validation fold contains both target classes.
✓ Validation periods do not overlap.
✓ Final 2025+ test period remains untouched.


## Historical-rate probability baseline

In [23]:
# Step 11.10 — Historical appearance-rate probability baseline

from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
)


baseline_rows = []
baseline_oof_parts = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[
        fold["train_idx"]
    ]

    val = dev_df.loc[
        fold["val_idx"]
    ]


    # Historical probability estimated only from training history
    historical_rate = train["appeared"].mean()

    val_prob = np.full(
        len(val),
        historical_rate,
        dtype=float,
    )

    y_val = val["appeared"].to_numpy()


    brier = brier_score_loss(
        y_val,
        val_prob,
    )

    logloss = log_loss(
        y_val,
        val_prob,
        labels=[0, 1],
    )


    baseline_rows.append({
        "fold": fold_number,
        "train_rate": historical_rate,
        "actual_val_rate": y_val.mean(),
        "brier": brier,
        "log_loss": logloss,
    })


    fold_predictions = pd.DataFrame({
        "row_index": val.index,
        "fold": fold_number,
        "actual": y_val,
        "predicted_probability": val_prob,
    })

    baseline_oof_parts.append(
        fold_predictions
    )


baseline_results = pd.DataFrame(
    baseline_rows
)

baseline_oof = (
    pd.concat(
        baseline_oof_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


# Pooled OOF metrics
oof_y = baseline_oof[
    "actual"
].to_numpy()

oof_prob = baseline_oof[
    "predicted_probability"
].to_numpy()


pooled_brier = brier_score_loss(
    oof_y,
    oof_prob,
)

pooled_logloss = log_loss(
    oof_y,
    oof_prob,
    labels=[0, 1],
)


print("HISTORICAL APPEARANCE-RATE BASELINE")
print("=" * 95)

print(
    baseline_results.to_string(
        index=False,
        formatters={
            "train_rate": "{:.4f}".format,
            "actual_val_rate": "{:.4f}".format,
            "brier": "{:.5f}".format,
            "log_loss": "{:.5f}".format,
        }
    )
)


print("\nMacro averages")
print("-" * 40)

print(
    "Brier:",
    round(
        baseline_results["brier"].mean(),
        6,
    ),
)

print(
    "Log loss:",
    round(
        baseline_results["log_loss"].mean(),
        6,
    ),
)


print("\nPooled OOF")
print("-" * 40)

print("Rows:", len(baseline_oof))
print(
    "Actual appearance rate:",
    round(oof_y.mean(), 4),
)

print(
    "Mean predicted probability:",
    round(oof_prob.mean(), 4),
)

print(
    "Brier:",
    round(pooled_brier, 6),
)

print(
    "Log loss:",
    round(pooled_logloss, 6),
)


assert len(baseline_oof) == 513
assert baseline_oof["row_index"].nunique() == 513

assert baseline_oof[
    "predicted_probability"
].between(0, 1).all()


print(
    "\n✓ Leakage-safe historical-rate baseline evaluated."
)

print(
    "✓ Baseline probabilities use training history only."
)

print(
    "✓ Final 2025+ test set remains untouched."
)

HISTORICAL APPEARANCE-RATE BASELINE
 fold train_rate actual_val_rate   brier log_loss
    1     0.8498          0.7793 0.17695  0.54521
    2     0.8406          0.8261 0.14388  0.46281
    3     0.8393          0.8538 0.12500  0.41679
    4     0.8407          0.8943 0.09740  0.34934

Macro averages
----------------------------------------
Brier: 0.135809
Log loss: 0.443537

Pooled OOF
----------------------------------------
Rows: 513
Actual appearance rate: 0.8363
Mean predicted probability: 0.8429
Brier: 0.1373
Log loss: 0.447232

✓ Leakage-safe historical-rate baseline evaluated.
✓ Baseline probabilities use training history only.
✓ Final 2025+ test set remains untouched.


In [25]:
# Build preprocessing for the appearance model

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


appearance_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
)


print("APPEARANCE MODEL — PREPROCESSING")
print("=" * 80)

print(
    "Numeric features:",
    len(numeric_features),
)

print(
    "Categorical features:",
    len(categorical_features),
)

print(
    "Total raw predictors:",
    len(appearance_features),
)


# Fit only on the earliest fold's training data
# as a structural sanity check.
first_fold = temporal_folds[0]

first_train = dev_df.loc[
    first_fold["train_idx"]
]

X_first_train = first_train[
    appearance_features
]


X_transformed = (
    appearance_preprocessor
    .fit_transform(X_first_train)
)


print(
    "\nEarliest training rows:",
    len(X_first_train),
)

print(
    "Transformed shape:",
    X_transformed.shape,
)


feature_names = (
    appearance_preprocessor
    .get_feature_names_out()
)

print(
    "Transformed predictors:",
    len(feature_names),
)


print("\nFirst 20 transformed feature names:")

for name in feature_names[:20]:
    print(" ", name)


assert X_transformed.shape[0] == len(first_train)

assert X_transformed.shape[1] == len(feature_names)

assert len(feature_names) >= len(appearance_features)


print(
    "\n✓ Numeric features standardized."
)

print(
    "✓ Categorical features one-hot encoded."
)

print(
    "✓ Unknown future categories will be handled safely."
)

print(
    "✓ Preprocessing structure verified."
)

APPEARANCE MODEL — PREPROCESSING
Numeric features: 12
Categorical features: 3
Total raw predictors: 15

Earliest training rows: 972
Transformed shape: (972, 39)
Transformed predictors: 39

First 20 transformed feature names:
  numeric__appeared_previous_pre
  numeric__appearance_rate_last_5_pre
  numeric__appearance_rate_last_10_pre
  numeric__consecutive_appearances_pre
  numeric__days_since_last_fixture_pre
  numeric__fixtures_last_7_days_pre
  numeric__fixtures_last_14_days_pre
  numeric__age_years_pre
  numeric__is_national_team
  numeric__fixture_month
  numeric__team_appeared_previous_pre
  numeric__team_appearance_rate_last_5_pre
  categorical__team_Manchester United
  categorical__team_Portugal
  categorical__team_Real Madrid
  categorical__team_Sporting CP
  categorical__competition_Champions League
  categorical__competition_Champions League Qualifying
  categorical__competition_Copa del Rey
  categorical__competition_Euro Championship

✓ Numeric features standardized.
✓ Cate

## Logistic regression OOF model

In [28]:
# Step 11.12 — Logistic regression with expanding temporal validation

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


logistic_rows = []
logistic_oof_parts = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[fold["train_idx"]]
    val = dev_df.loc[fold["val_idx"]]

    X_train = train[appearance_features]
    y_train = train[target]

    X_val = val[appearance_features]
    y_val = val[target]


    # Fresh preprocessing + model for every fold
    model = Pipeline([
        (
            "preprocessor",
            clone(appearance_preprocessor),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                max_iter=2000,
                solver="lbfgs",
            ),
        ),
    ])


    model.fit(
        X_train,
        y_train,
    )


    val_prob = model.predict_proba(
        X_val
    )[:, 1]


    brier = brier_score_loss(
        y_val,
        val_prob,
    )

    logloss = log_loss(
        y_val,
        val_prob,
        labels=[0, 1],
    )

    auc = roc_auc_score(
        y_val,
        val_prob,
    )


    logistic_rows.append({
        "fold": fold_number,
        "actual_rate": y_val.mean(),
        "predicted_rate": val_prob.mean(),
        "brier": brier,
        "log_loss": logloss,
        "roc_auc": auc,
    })


    fold_predictions = pd.DataFrame({
        "row_index": val.index,
        "fold": fold_number,
        "actual": y_val.to_numpy(),
        "predicted_probability": val_prob,
    })

    logistic_oof_parts.append(
        fold_predictions
    )


logistic_results = pd.DataFrame(
    logistic_rows
)

logistic_oof = (
    pd.concat(
        logistic_oof_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


# Pooled OOF metrics
oof_y = logistic_oof[
    "actual"
].to_numpy()

oof_prob = logistic_oof[
    "predicted_probability"
].to_numpy()


pooled_brier = brier_score_loss(
    oof_y,
    oof_prob,
)

pooled_logloss = log_loss(
    oof_y,
    oof_prob,
    labels=[0, 1],
)

pooled_auc = roc_auc_score(
    oof_y,
    oof_prob,
)


print("LOGISTIC REGRESSION — TEMPORAL OOF")
print("=" * 105)

print(
    logistic_results.to_string(
        index=False,
        formatters={
            "actual_rate": "{:.4f}".format,
            "predicted_rate": "{:.4f}".format,
            "brier": "{:.5f}".format,
            "log_loss": "{:.5f}".format,
            "roc_auc": "{:.4f}".format,
        }
    )
)


print("\nMacro averages")
print("-" * 45)

print(
    "Brier:",
    round(
        logistic_results["brier"].mean(),
        6,
    ),
)

print(
    "Log loss:",
    round(
        logistic_results["log_loss"].mean(),
        6,
    ),
)

print(
    "ROC AUC:",
    round(
        logistic_results["roc_auc"].mean(),
        6,
    ),
)


print("\nPooled OOF")
print("-" * 45)

print("Rows:", len(logistic_oof))

print(
    "Actual appearance rate:",
    round(oof_y.mean(), 4),
)

print(
    "Mean predicted probability:",
    round(oof_prob.mean(), 4),
)

print(
    "Brier:",
    round(pooled_brier, 6),
)

print(
    "Log loss:",
    round(pooled_logloss, 6),
)

print(
    "ROC AUC:",
    round(pooled_auc, 6),
)


print("\nHistorical-rate baseline")
print("-" * 45)

print("Brier:   0.137300")
print("Log loss: 0.447232")


assert len(logistic_oof) == 513

assert (
    logistic_oof["row_index"]
    .nunique()
    == 513
)

assert (
    logistic_oof[
        "predicted_probability"
    ]
    .between(0, 1)
    .all()
)


print(
    "\n✓ Logistic regression evaluated on all four temporal folds."
)

print(
    "✓ Every fold refits preprocessing from historical training data only."
)

print(
    "✓ 2025+ final test remains untouched."
)

LOGISTIC REGRESSION — TEMPORAL OOF
 fold actual_rate predicted_rate   brier log_loss roc_auc
    1      0.7793         0.8192 0.18133  0.57700  0.5655
    2      0.8261         0.8954 0.17910  0.63348  0.5416
    3      0.8538         0.7856 0.12772  0.41326  0.6766
    4      0.8943         0.7778 0.10893  0.36920  0.7867

Macro averages
---------------------------------------------
Brier: 0.149268
Log loss: 0.498235
ROC AUC: 0.642615

Pooled OOF
---------------------------------------------
Rows: 513
Actual appearance rate: 0.8363
Mean predicted probability: 0.8178
Brier: 0.149884
Log loss: 0.498344
ROC AUC: 0.560773

Historical-rate baseline
---------------------------------------------
Brier:   0.137300
Log loss: 0.447232

✓ Logistic regression evaluated on all four temporal folds.
✓ Every fold refits preprocessing from historical training data only.
✓ 2025+ final test remains untouched.


In [30]:
# Audit multiple eligible opportunities on the same date

same_date_counts = (
    df.groupby("date")
    .size()
    .sort_values(ascending=False)
)

duplicate_dates = same_date_counts[
    same_date_counts > 1
].index


same_date_rows = (
    df.loc[
        df["date"].isin(duplicate_dates),
        [
            "date",
            "team",
            "opponent",
            "competition",
            "venue",
            "appeared",
            "canonical_match_id",
        ],
    ]
    .sort_values(
        ["date", "team", "opponent"]
    )
    .reset_index(drop=True)
)


print("SAME-DATE ELIGIBLE OPPORTUNITY AUDIT")
print("=" * 100)

print(
    "Dates with multiple opportunities:",
    len(duplicate_dates),
)

print(
    "Rows on those dates:",
    len(same_date_rows),
)

print(
    "Maximum opportunities on one date:",
    same_date_counts.max(),
)


if len(same_date_rows) > 0:

    print("\nAffected rows")
    print("-" * 100)

    print(
        same_date_rows.to_string(
            index=False
        )
    )

else:

    print(
        "\n✓ No same-date duplicate opportunities found."
    )


print("\nCross-team duplicate dates")
print("-" * 100)

cross_team_dates = []

for date in duplicate_dates:

    date_rows = df[
        df["date"].eq(date)
    ]

    if date_rows["team"].nunique() > 1:
        cross_team_dates.append(date)


print(
    "Cross-team dates:",
    len(cross_team_dates)
)

if cross_team_dates:

    print(
        df.loc[
            df["date"].isin(cross_team_dates),
            [
                "date",
                "team",
                "opponent",
                "appeared",
            ],
        ]
        .sort_values(["date", "team"])
        .to_string(index=False)
    )


print(
    "\n✓ Same-date audit complete."
)

SAME-DATE ELIGIBLE OPPORTUNITY AUDIT
Dates with multiple opportunities: 1
Rows on those dates: 2
Maximum opportunities on one date: 2

Affected rows
----------------------------------------------------------------------------------------------------
      date        team opponent            competition venue  appeared                  canonical_match_id
2017-06-03    Portugal   Cyprus International Friendly     H         0                                 NaN
2017-06-03 Real Madrid Juventus       Champions League     A         1 2017-06-03_Real_Madrid_Juventus_855

Cross-team duplicate dates
----------------------------------------------------------------------------------------------------
Cross-team dates: 1
      date        team opponent  appeared
2017-06-03    Portugal   Cyprus         0
2017-06-03 Real Madrid Juventus         1

✓ Same-date audit complete.


In [32]:
# Rebuild global history features using strictly earlier dates

from collections import deque


df = df.sort_values(
    ["date", "team", "opponent"]
).reset_index(drop=True)


# Store completed eligible opportunities from strictly earlier dates.
prior_outcomes = []
prior_dates = []

new_previous = []
new_rate_5 = []
new_rate_10 = []
new_streak = []
new_days_since = []


for current_date, date_group in df.groupby("date", sort=True):

    # All rows on the same date receive the same global history state.
    if prior_outcomes:

        previous = prior_outcomes[-1]

        rate_5 = np.mean(
            prior_outcomes[-5:]
        )

        rate_10 = np.mean(
            prior_outcomes[-10:]
        )

        streak = 0

        for outcome in reversed(prior_outcomes):
            if outcome == 1:
                streak += 1
            else:
                break

        days_since = (
            current_date - prior_dates[-1]
        ).days

    else:

        previous = 0
        rate_5 = 0.0
        rate_10 = 0.0
        streak = 0
        days_since = 0


    n_rows = len(date_group)

    new_previous.extend(
        [previous] * n_rows
    )

    new_rate_5.extend(
        [rate_5] * n_rows
    )

    new_rate_10.extend(
        [rate_10] * n_rows
    )

    new_streak.extend(
        [streak] * n_rows
    )

    new_days_since.extend(
        [days_since] * n_rows
    )


    # Only after every row on this date has received its features
    # do today's outcomes become historical information.
    prior_outcomes.extend(
        date_group["appeared"].astype(int).tolist()
    )

    prior_dates.extend(
        [current_date] * n_rows
    )


df["appeared_previous_pre"] = new_previous
df["appearance_rate_last_5_pre"] = new_rate_5
df["appearance_rate_last_10_pre"] = new_rate_10
df["consecutive_appearances_pre"] = new_streak
df["days_since_last_fixture_pre"] = new_days_since


# Recreate temporal holdout after correcting the feature values.
dev_df = (
    df.loc[df["date"] < TEST_START]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    df.loc[df["date"] >= TEST_START]
    .copy()
    .reset_index(drop=True)
)


print("STRICT PRIOR-DATE FEATURE CORRECTION")
print("=" * 90)

check = df.loc[
    df["date"].eq("2017-06-03"),
    [
        "date",
        "team",
        "opponent",
        "appeared",
        "appeared_previous_pre",
        "appearance_rate_last_5_pre",
        "appearance_rate_last_10_pre",
        "consecutive_appearances_pre",
        "days_since_last_fixture_pre",
    ],
]

print(check.to_string(index=False))


# Same-date rows must now have identical global pre-fixture state.
global_cols = [
    "appeared_previous_pre",
    "appearance_rate_last_5_pre",
    "appearance_rate_last_10_pre",
    "consecutive_appearances_pre",
    "days_since_last_fixture_pre",
]

assert (
    check[global_cols]
    .nunique()
    .eq(1)
    .all()
)

assert len(dev_df) == 1485
assert len(test_df) == 99

assert df[appearance_features].isna().sum().sum() == 0


print(
    "\n✓ Same-date fixtures can no longer influence one another."
)

print(
    "✓ Global history now uses strictly earlier calendar dates."
)

print(
    "✓ Feature set remains 15 predictors."
)

print(
    "✓ Development/test boundary remains unchanged."
)

STRICT PRIOR-DATE FEATURE CORRECTION
      date        team opponent  appeared  appeared_previous_pre  appearance_rate_last_5_pre  appearance_rate_last_10_pre  consecutive_appearances_pre  days_since_last_fixture_pre
2017-06-03    Portugal   Cyprus         0                      1                         0.8                          0.8                            4                           13
2017-06-03 Real Madrid Juventus         1                      1                         0.8                          0.8                            4                           13

✓ Same-date fixtures can no longer influence one another.
✓ Global history now uses strictly earlier calendar dates.
✓ Feature set remains 15 predictors.
✓ Development/test boundary remains unchanged.


In [34]:
# Rebuild temporal folds after strict-date correction

fold_specs = [
    {"fold": 1, "val_start": "2017-01-01", "val_end": "2018-12-31"},
    {"fold": 2, "val_start": "2019-01-01", "val_end": "2020-12-31"},
    {"fold": 3, "val_start": "2021-01-01", "val_end": "2022-12-31"},
    {"fold": 4, "val_start": "2023-01-01", "val_end": "2024-12-31"},
]

temporal_folds = []
audit_rows = []

for spec in fold_specs:

    val_start = pd.Timestamp(spec["val_start"])
    val_end = pd.Timestamp(spec["val_end"])

    train_idx = dev_df.index[
        dev_df["date"] < val_start
    ].tolist()

    val_idx = dev_df.index[
        (dev_df["date"] >= val_start)
        & (dev_df["date"] <= val_end)
    ].tolist()

    train = dev_df.loc[train_idx]
    val = dev_df.loc[val_idx]

    assert len(train) > 0
    assert len(val) > 0
    assert train["date"].max() < val["date"].min()
    assert val["appeared"].nunique() == 2

    temporal_folds.append({
        "fold": spec["fold"],
        "train_idx": train_idx,
        "val_idx": val_idx,
    })

    audit_rows.append({
        "fold": spec["fold"],
        "train_n": len(train),
        "train_apps": int(train["appeared"].sum()),
        "train_nonapps": int(train["appeared"].eq(0).sum()),
        "val_n": len(val),
        "val_apps": int(val["appeared"].sum()),
        "val_nonapps": int(val["appeared"].eq(0).sum()),
        "val_rate": val["appeared"].mean(),
        "train_end": train["date"].max().date(),
        "val_start": val["date"].min().date(),
        "val_end": val["date"].max().date(),
    })


fold_audit_corrected = pd.DataFrame(audit_rows)

print("CORRECTED APPEARANCE MODEL — TEMPORAL FOLDS")
print("=" * 115)

print(
    fold_audit_corrected.to_string(
        index=False,
        formatters={
            "val_rate": "{:.4f}".format
        }
    )
)


all_val_idx = [
    idx
    for fold in temporal_folds
    for idx in fold["val_idx"]
]

print("\nValidation coverage")
print("-" * 50)

print("OOF rows:", len(all_val_idx))
print("Unique OOF rows:", len(set(all_val_idx)))

print(
    "OOF appearances:",
    int(dev_df.loc[all_val_idx, "appeared"].sum()),
)

print(
    "OOF nonappearances:",
    int(dev_df.loc[all_val_idx, "appeared"].eq(0).sum()),
)


assert len(all_val_idx) == 513
assert len(set(all_val_idx)) == 513

assert dev_df.loc[
    all_val_idx,
    "appeared"
].sum() == 429

assert dev_df.loc[
    all_val_idx,
    "appeared"
].eq(0).sum() == 84

assert dev_df["date"].max() < TEST_START
assert test_df["date"].min() >= TEST_START


print("\n✓ Temporal folds rebuilt from corrected feature data.")
print("✓ 513 OOF fixtures and 84 nonappearances preserved.")
print("✓ 2025+ final test remains untouched.")

CORRECTED APPEARANCE MODEL — TEMPORAL FOLDS
 fold  train_n  train_apps  train_nonapps  val_n  val_apps  val_nonapps val_rate  train_end  val_start    val_end
    1      972         826            146    145       113           32   0.7793 2016-12-18 2017-01-04 2018-12-29
    2     1117         939            178    115        95           20   0.8261 2018-12-29 2019-01-12 2020-12-22
    3     1232        1034            198    130       111           19   0.8538 2020-12-22 2021-01-03 2022-12-10
    4     1362        1145            217    123       110           13   0.8943 2022-12-10 2023-01-22 2024-12-06

Validation coverage
--------------------------------------------------
OOF rows: 513
Unique OOF rows: 513
OOF appearances: 429
OOF nonappearances: 84

✓ Temporal folds rebuilt from corrected feature data.
✓ 513 OOF fixtures and 84 nonappearances preserved.
✓ 2025+ final test remains untouched.


In [36]:
# CatBoost appearance model with temporal validation

from catboost import CatBoostClassifier


catboost_rows = []
catboost_oof_parts = []

cat_features = categorical_features


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[fold["train_idx"]].copy()
    val = dev_df.loc[fold["val_idx"]].copy()

    X_train = train[appearance_features].copy()
    y_train = train[target]

    X_val = val[appearance_features].copy()
    y_val = val[target]

    # CatBoost expects categorical columns to contain strings.
    for col in cat_features:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)

    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=250,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=10,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
    )

    val_prob = model.predict_proba(X_val)[:, 1]

    brier = brier_score_loss(
        y_val,
        val_prob,
    )

    logloss = log_loss(
        y_val,
        val_prob,
        labels=[0, 1],
    )

    auc = roc_auc_score(
        y_val,
        val_prob,
    )

    catboost_rows.append({
        "fold": fold_number,
        "actual_rate": y_val.mean(),
        "predicted_rate": val_prob.mean(),
        "brier": brier,
        "log_loss": logloss,
        "roc_auc": auc,
    })

    fold_predictions = pd.DataFrame({
        "row_index": val.index,
        "fold": fold_number,
        "actual": y_val.to_numpy(),
        "predicted_probability": val_prob,
    })

    catboost_oof_parts.append(
        fold_predictions
    )


catboost_results = pd.DataFrame(
    catboost_rows
)

catboost_oof = (
    pd.concat(
        catboost_oof_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


# Pooled OOF metrics
oof_y = catboost_oof[
    "actual"
].to_numpy()

oof_prob = catboost_oof[
    "predicted_probability"
].to_numpy()


pooled_brier = brier_score_loss(
    oof_y,
    oof_prob,
)

pooled_logloss = log_loss(
    oof_y,
    oof_prob,
    labels=[0, 1],
)

pooled_auc = roc_auc_score(
    oof_y,
    oof_prob,
)


print("CATBOOST APPEARANCE MODEL — TEMPORAL OOF")
print("=" * 105)

print(
    catboost_results.to_string(
        index=False,
        formatters={
            "actual_rate": "{:.4f}".format,
            "predicted_rate": "{:.4f}".format,
            "brier": "{:.5f}".format,
            "log_loss": "{:.5f}".format,
            "roc_auc": "{:.4f}".format,
        }
    )
)


print("\nMacro averages")
print("-" * 45)

print(
    "Brier:",
    round(
        catboost_results["brier"].mean(),
        6,
    ),
)

print(
    "Log loss:",
    round(
        catboost_results["log_loss"].mean(),
        6,
    ),
)

print(
    "ROC AUC:",
    round(
        catboost_results["roc_auc"].mean(),
        6,
    ),
)


print("\nPooled OOF")
print("-" * 45)

print("Rows:", len(catboost_oof))

print(
    "Actual appearance rate:",
    round(oof_y.mean(), 4),
)

print(
    "Mean predicted probability:",
    round(oof_prob.mean(), 4),
)

print(
    "Brier:",
    round(pooled_brier, 6),
)

print(
    "Log loss:",
    round(pooled_logloss, 6),
)

print(
    "ROC AUC:",
    round(pooled_auc, 6),
)


print("\nReference models")
print("-" * 45)

print("Historical-rate baseline")
print("  Brier:   0.137300")
print("  Log loss: 0.447232")

print("\nLogistic regression")
print("  Brier:   0.149884")
print("  Log loss: 0.498344")


assert len(catboost_oof) == 513

assert (
    catboost_oof["row_index"]
    .nunique()
    == 513
)

assert (
    catboost_oof[
        "predicted_probability"
    ]
    .between(0, 1)
    .all()
)


print(
    "\n✓ CatBoost evaluated on all four corrected temporal folds."
)

print(
    "✓ Categorical predictors handled natively."
)

print(
    "✓ Final 2025+ test remains untouched."
)

CATBOOST APPEARANCE MODEL — TEMPORAL OOF
 fold actual_rate predicted_rate   brier log_loss roc_auc
    1      0.7793         0.8293 0.16449  0.52606  0.5788
    2      0.8261         0.8537 0.16263  0.51901  0.5511
    3      0.8538         0.8387 0.12023  0.40197  0.6017
    4      0.8943         0.8085 0.10209  0.35205  0.7353

Macro averages
---------------------------------------------
Brier: 0.137361
Log loss: 0.44977
ROC AUC: 0.616723

Pooled OOF
---------------------------------------------
Rows: 513
Actual appearance rate: 0.8363
Mean predicted probability: 0.8322
Brier: 0.137897
Log loss: 0.45131
ROC AUC: 0.576854

Reference models
---------------------------------------------
Historical-rate baseline
  Brier:   0.137300
  Log loss: 0.447232

Logistic regression
  Brier:   0.149884
  Log loss: 0.498344

✓ CatBoost evaluated on all four corrected temporal folds.
✓ Categorical predictors handled natively.
✓ Final 2025+ test remains untouched.


In [38]:
# Inspect CatBoost feature importance on development data

X_dev = dev_df[
    appearance_features
].copy()

y_dev = dev_df[
    target
].copy()


for col in categorical_features:
    X_dev[col] = X_dev[col].astype(str)


diagnostic_model = CatBoostClassifier(
    loss_function="Logloss",
    iterations=250,
    depth=3,
    learning_rate=0.03,
    l2_leaf_reg=10,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)


diagnostic_model.fit(
    X_dev,
    y_dev,
    cat_features=categorical_features,
)


feature_importance = pd.DataFrame({
    "feature": appearance_features,
    "importance": diagnostic_model.get_feature_importance(),
})

feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)


feature_importance["importance_pct"] = (
    feature_importance["importance"]
    / feature_importance["importance"].sum()
    * 100
)


print("CATBOOST — APPEARANCE FEATURE IMPORTANCE")
print("=" * 80)

print(
    feature_importance.to_string(
        index=False,
        formatters={
            "importance": "{:.4f}".format,
            "importance_pct": "{:.2f}%".format,
        }
    )
)


print("\nTop-feature concentration")
print("-" * 50)

print(
    "Top 3:",
    f"{feature_importance.head(3)['importance_pct'].sum():.2f}%"
)

print(
    "Top 5:",
    f"{feature_importance.head(5)['importance_pct'].sum():.2f}%"
)


assert len(feature_importance) == 15

assert np.isclose(
    feature_importance["importance"].sum(),
    100.0,
)


print(
    "\n✓ Feature importance calculated using development data only."
)

print(
    "✓ 2025+ final test remains completely untouched."
)

CATBOOST — APPEARANCE FEATURE IMPORTANCE
                        feature importance importance_pct
                    competition    32.2573         32.26%
          appeared_previous_pre    11.5450         11.55%
                  age_years_pre     9.7870          9.79%
                           team     9.3575          9.36%
                  fixture_month     8.7766          8.78%
     appearance_rate_last_5_pre     8.6991          8.70%
    days_since_last_fixture_pre     4.6272          4.63%
      fixtures_last_14_days_pre     3.1597          3.16%
     team_appeared_previous_pre     2.7787          2.78%
    consecutive_appearances_pre     2.1450          2.14%
                          venue     1.8785          1.88%
    appearance_rate_last_10_pre     1.5551          1.56%
team_appearance_rate_last_5_pre     1.5162          1.52%
               is_national_team     1.0382          1.04%
       fixtures_last_7_days_pre     0.8790          0.88%

Top-feature concentration
----

In [40]:
# CatBoost ablation without team and competition

ablation_features = [
    feature
    for feature in appearance_features
    if feature not in ["team", "competition"]
]

ablation_cat_features = [
    "venue"
]


ablation_rows = []
ablation_oof_parts = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[
        fold["train_idx"]
    ].copy()

    val = dev_df.loc[
        fold["val_idx"]
    ].copy()


    X_train = train[
        ablation_features
    ].copy()

    y_train = train[target]

    X_val = val[
        ablation_features
    ].copy()

    y_val = val[target]


    for col in ablation_cat_features:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)


    model = CatBoostClassifier(
        loss_function="Logloss",
        iterations=250,
        depth=3,
        learning_rate=0.03,
        l2_leaf_reg=10,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
    )


    model.fit(
        X_train,
        y_train,
        cat_features=ablation_cat_features,
    )


    val_prob = model.predict_proba(
        X_val
    )[:, 1]


    ablation_rows.append({
        "fold": fold_number,
        "actual_rate": y_val.mean(),
        "predicted_rate": val_prob.mean(),

        "brier": brier_score_loss(
            y_val,
            val_prob,
        ),

        "log_loss": log_loss(
            y_val,
            val_prob,
            labels=[0, 1],
        ),

        "roc_auc": roc_auc_score(
            y_val,
            val_prob,
        ),
    })


    ablation_oof_parts.append(
        pd.DataFrame({
            "row_index": val.index,
            "fold": fold_number,
            "actual": y_val.to_numpy(),
            "predicted_probability": val_prob,
        })
    )


ablation_results = pd.DataFrame(
    ablation_rows
)


ablation_oof = (
    pd.concat(
        ablation_oof_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


oof_y = ablation_oof[
    "actual"
].to_numpy()

oof_prob = ablation_oof[
    "predicted_probability"
].to_numpy()


pooled_brier = brier_score_loss(
    oof_y,
    oof_prob,
)

pooled_logloss = log_loss(
    oof_y,
    oof_prob,
    labels=[0, 1],
)

pooled_auc = roc_auc_score(
    oof_y,
    oof_prob,
)


print("CATBOOST ABLATION — NO TEAM / COMPETITION")
print("=" * 105)

print(
    ablation_results.to_string(
        index=False,
        formatters={
            "actual_rate": "{:.4f}".format,
            "predicted_rate": "{:.4f}".format,
            "brier": "{:.5f}".format,
            "log_loss": "{:.5f}".format,
            "roc_auc": "{:.4f}".format,
        }
    )
)


print("\nPooled OOF")
print("-" * 45)

print("Predictors:", len(ablation_features))
print("Rows:", len(ablation_oof))

print(
    "Actual appearance rate:",
    round(oof_y.mean(), 4),
)

print(
    "Mean predicted probability:",
    round(oof_prob.mean(), 4),
)

print(
    "Brier:",
    round(pooled_brier, 6),
)

print(
    "Log loss:",
    round(pooled_logloss, 6),
)

print(
    "ROC AUC:",
    round(pooled_auc, 6),
)


print("\nCurrent comparison")
print("-" * 45)

print("Historical-rate baseline")
print("  Brier:    0.137300")
print("  Log loss: 0.447232")

print("\nFull CatBoost")
print("  Brier:    0.137897")
print("  Log loss: 0.451310")

print("\nNo team/competition CatBoost")
print(
    "  Brier:   ",
    f"{pooled_brier:.6f}",
)

print(
    "  Log loss:",
    f"{pooled_logloss:.6f}",
)


assert len(ablation_features) == 13
assert "team" not in ablation_features
assert "competition" not in ablation_features

assert len(ablation_oof) == 513
assert ablation_oof["row_index"].nunique() == 513


print(
    "\n✓ Ablation evaluated on the same temporal OOF fixtures."
)

print(
    "✓ No changes made to the frozen 2025+ final test."
)

CATBOOST ABLATION — NO TEAM / COMPETITION
 fold actual_rate predicted_rate   brier log_loss roc_auc
    1      0.7793         0.8309 0.17919  0.56834  0.4313
    2      0.8261         0.8110 0.14869  0.46995  0.5900
    3      0.8538         0.8260 0.12633  0.42017  0.5322
    4      0.8943         0.8423 0.09656  0.34405  0.5787

Pooled OOF
---------------------------------------------
Predictors: 13
Rows: 513
Actual appearance rate: 0.8363
Mean predicted probability: 0.8279
Brier: 0.139143
Log loss: 0.454957
ROC AUC: 0.501485

Current comparison
---------------------------------------------
Historical-rate baseline
  Brier:    0.137300
  Log loss: 0.447232

Full CatBoost
  Brier:    0.137897
  Log loss: 0.451310

No team/competition CatBoost
  Brier:    0.139143
  Log loss: 0.454957

✓ Ablation evaluated on the same temporal OOF fixtures.
✓ No changes made to the frozen 2025+ final test.


In [42]:
# Empirical signal in recent appearance-history features

signal_rows = []


# Previous eligible fixture
previous_summary = (
    dev_df
    .groupby("appeared_previous_pre")["appeared"]
    .agg(
        opportunities="size",
        appearances="sum",
        appearance_rate="mean",
    )
    .reset_index()
)

print("APPEARANCE RATE BY PREVIOUS FIXTURE STATUS")
print("=" * 75)

print(
    previous_summary.to_string(
        index=False,
        formatters={
            "appearance_rate": "{:.4f}".format
        }
    )
)


# Recent 5-fixture appearance rate
dev_df["recent_5_band"] = pd.cut(
    dev_df["appearance_rate_last_5_pre"],
    bins=[
        -0.01,
        0.59,
        0.79,
        0.99,
        1.01,
    ],
    labels=[
        "< 60%",
        "60–79%",
        "80–99%",
        "100%",
    ],
)


recent_summary = (
    dev_df
    .groupby(
        "recent_5_band",
        observed=True,
    )["appeared"]
    .agg(
        opportunities="size",
        appearances="sum",
        appearance_rate="mean",
    )
    .reset_index()
)


print("\nAPPEARANCE RATE BY PRIOR 5-FIXTURE HISTORY")
print("=" * 75)

print(
    recent_summary.to_string(
        index=False,
        formatters={
            "appearance_rate": "{:.4f}".format
        }
    )
)


# Current appearance streak
dev_df["streak_band"] = pd.cut(
    dev_df["consecutive_appearances_pre"],
    bins=[
        -1,
        0,
        2,
        5,
        10,
        float("inf"),
    ],
    labels=[
        "0",
        "1–2",
        "3–5",
        "6–10",
        "11+",
    ],
)


streak_summary = (
    dev_df
    .groupby(
        "streak_band",
        observed=True,
    )["appeared"]
    .agg(
        opportunities="size",
        appearances="sum",
        appearance_rate="mean",
    )
    .reset_index()
)


print("\nAPPEARANCE RATE BY CURRENT APPEARANCE STREAK")
print("=" * 75)

print(
    streak_summary.to_string(
        index=False,
        formatters={
            "appearance_rate": "{:.4f}".format
        }
    )
)


# Clean temporary diagnostic columns.
dev_df.drop(
    columns=[
        "recent_5_band",
        "streak_band",
    ],
    inplace=True,
)


print(
    "\n✓ Diagnostics use development data only."
)

print(
    "✓ No model was trained and the 2025+ test remains untouched."
)

APPEARANCE RATE BY PREVIOUS FIXTURE STATUS
 appeared_previous_pre  opportunities  appearances appearance_rate
                     0            230          157          0.6826
                     1           1255         1098          0.8749

APPEARANCE RATE BY PRIOR 5-FIXTURE HISTORY
recent_5_band  opportunities  appearances appearance_rate
        < 60%             93           55          0.5914
       60–79%            172          141          0.8198
       80–99%            478          410          0.8577
         100%            742          649          0.8747

APPEARANCE RATE BY CURRENT APPEARANCE STREAK
streak_band  opportunities  appearances appearance_rate
          0            230          157          0.6826
        1–2            294          255          0.8673
        3–5            312          278          0.8910
       6–10            300          252          0.8400
        11+            349          313          0.8968

✓ Diagnostics use development data only

In [44]:
# Historical conditional probability by previous appearance state

state_rows = []
state_oof_parts = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[
        fold["train_idx"]
    ].copy()

    val = dev_df.loc[
        fold["val_idx"]
    ].copy()


    # Estimate probabilities from historical training data only.
    state_probabilities = (
        train
        .groupby("appeared_previous_pre")["appeared"]
        .mean()
        .to_dict()
    )

    fallback_probability = train[
        "appeared"
    ].mean()


    val_prob = (
        val["appeared_previous_pre"]
        .map(state_probabilities)
        .fillna(fallback_probability)
        .to_numpy()
    )

    y_val = val[
        target
    ].to_numpy()


    state_rows.append({
        "fold": fold_number,

        "p_after_nonappearance":
            state_probabilities.get(
                0,
                fallback_probability,
            ),

        "p_after_appearance":
            state_probabilities.get(
                1,
                fallback_probability,
            ),

        "actual_rate":
            y_val.mean(),

        "predicted_rate":
            val_prob.mean(),

        "brier":
            brier_score_loss(
                y_val,
                val_prob,
            ),

        "log_loss":
            log_loss(
                y_val,
                val_prob,
                labels=[0, 1],
            ),

        "roc_auc":
            roc_auc_score(
                y_val,
                val_prob,
            ),
    })


    state_oof_parts.append(
        pd.DataFrame({
            "row_index": val.index,
            "fold": fold_number,
            "actual": y_val,
            "predicted_probability": val_prob,
        })
    )


state_results = pd.DataFrame(
    state_rows
)


state_oof = (
    pd.concat(
        state_oof_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


oof_y = state_oof[
    "actual"
].to_numpy()

oof_prob = state_oof[
    "predicted_probability"
].to_numpy()


pooled_brier = brier_score_loss(
    oof_y,
    oof_prob,
)

pooled_logloss = log_loss(
    oof_y,
    oof_prob,
    labels=[0, 1],
)

pooled_auc = roc_auc_score(
    oof_y,
    oof_prob,
)


print("PREVIOUS-APPEARANCE STATE BASELINE — TEMPORAL OOF")
print("=" * 110)

print(
    state_results.to_string(
        index=False,
        formatters={
            "p_after_nonappearance": "{:.4f}".format,
            "p_after_appearance": "{:.4f}".format,
            "actual_rate": "{:.4f}".format,
            "predicted_rate": "{:.4f}".format,
            "brier": "{:.5f}".format,
            "log_loss": "{:.5f}".format,
            "roc_auc": "{:.4f}".format,
        }
    )
)


print("\nPooled OOF")
print("-" * 50)

print(
    "Actual appearance rate:",
    round(oof_y.mean(), 4),
)

print(
    "Mean predicted probability:",
    round(oof_prob.mean(), 4),
)

print(
    "Brier:",
    round(pooled_brier, 6),
)

print(
    "Log loss:",
    round(pooled_logloss, 6),
)

print(
    "ROC AUC:",
    round(pooled_auc, 6),
)


print("\nCurrent comparison")
print("-" * 50)

print("Historical-rate baseline")
print("  Brier:    0.137300")
print("  Log loss: 0.447232")

print("\nFull CatBoost")
print("  Brier:    0.137897")
print("  Log loss: 0.451310")

print("\nPrevious-appearance state baseline")
print(
    "  Brier:   ",
    f"{pooled_brier:.6f}",
)
print(
    "  Log loss:",
    f"{pooled_logloss:.6f}",
)


assert len(state_oof) == 513
assert state_oof["row_index"].nunique() == 513

assert (
    state_oof["predicted_probability"]
    .between(0, 1)
    .all()
)


print(
    "\n✓ State probabilities learned independently inside each temporal fold."
)

print(
    "✓ 2025+ final test remains untouched."
)

PREVIOUS-APPEARANCE STATE BASELINE — TEMPORAL OOF
 fold p_after_nonappearance p_after_appearance actual_rate predicted_rate   brier log_loss roc_auc
    1                0.6190             0.8909      0.7793         0.8328 0.19042  0.59000  0.4831
    2                0.6517             0.8765      0.8261         0.8374 0.14120  0.45461  0.5763
    3                0.6566             0.8743      0.8538         0.8425 0.12674  0.41992  0.5377
    4                0.6682             0.8734      0.8943         0.8517 0.10156  0.35779  0.4839

Pooled OOF
--------------------------------------------------
Actual appearance rate: 0.8363
Mean predicted probability: 0.8408
Brier: 0.141945
Log loss: 0.460873
ROC AUC: 0.464702

Current comparison
--------------------------------------------------
Historical-rate baseline
  Brier:    0.137300
  Log loss: 0.447232

Full CatBoost
  Brier:    0.137897
  Log loss: 0.451310

Previous-appearance state baseline
  Brier:    0.141945
  Log loss: 0.460873


In [46]:
# Recency-based appearance probability baselines

window_sizes = [
    10,
    20,
    30,
    50,
    100,
]

rolling_rows = []
rolling_oof_predictions = {}


for window in window_sizes:

    fold_predictions = []

    for fold in temporal_folds:

        val = dev_df.loc[
            fold["val_idx"]
        ].copy()

        train = dev_df.loc[
            fold["train_idx"]
        ].copy()

        # Full historical sequence available before validation begins.
        history = train[
            ["date", "appeared"]
        ].copy()

        val_probs = []


        for idx, row in val.iterrows():

            current_date = row["date"]

            # Strictly earlier dates only.
            eligible_history = history.loc[
                history["date"] < current_date,
                "appeared",
            ]

            if len(eligible_history) == 0:
                probability = train[
                    "appeared"
                ].mean()

            else:
                probability = (
                    eligible_history
                    .tail(window)
                    .mean()
                )

            val_probs.append(
                probability
            )

            # Current result becomes history only after prediction.
            history = pd.concat(
                [
                    history,
                    pd.DataFrame({
                        "date": [current_date],
                        "appeared": [row["appeared"]],
                    }),
                ],
                ignore_index=True,
            )


        fold_predictions.append(
            pd.DataFrame({
                "row_index": val.index,
                "fold": fold["fold"],
                "actual": val[target].to_numpy(),
                "predicted_probability": val_probs,
            })
        )


    oof = (
        pd.concat(
            fold_predictions,
            ignore_index=True,
        )
        .sort_values("row_index")
        .reset_index(drop=True)
    )


    y_true = oof[
        "actual"
    ].to_numpy()

    y_prob = oof[
        "predicted_probability"
    ].to_numpy()


    rolling_rows.append({
        "window": window,
        "mean_probability": y_prob.mean(),

        "brier": brier_score_loss(
            y_true,
            y_prob,
        ),

        "log_loss": log_loss(
            y_true,
            y_prob,
            labels=[0, 1],
        ),

        "roc_auc": roc_auc_score(
            y_true,
            y_prob,
        ),
    })


    rolling_oof_predictions[
        window
    ] = oof


rolling_results = (
    pd.DataFrame(
        rolling_rows
    )
    .sort_values("brier")
    .reset_index(drop=True)
)


print("ROLLING APPEARANCE-RATE BASELINES")
print("=" * 85)

print(
    rolling_results.to_string(
        index=False,
        formatters={
            "mean_probability": "{:.4f}".format,
            "brier": "{:.6f}".format,
            "log_loss": "{:.6f}".format,
            "roc_auc": "{:.4f}".format,
        }
    )
)


print("\nExisting benchmarks")
print("-" * 55)

print(
    "Historical rate : Brier 0.137300 | Log loss 0.447232"
)

print(
    "Full CatBoost   : Brier 0.137897 | Log loss 0.451310"
)

print(
    "Logistic        : Brier 0.149884 | Log loss 0.498344"
)

print(
    "Previous state  : Brier 0.141945 | Log loss 0.460873"
)


best_window = int(
    rolling_results.iloc[0]["window"]
)

best_brier = float(
    rolling_results.iloc[0]["brier"]
)


print("\nBest rolling window")
print("-" * 55)

print(
    "Window:",
    best_window,
)

print(
    "Brier:",
    round(best_brier, 6),
)


assert all(
    len(oof) == 513
    for oof in rolling_oof_predictions.values()
)

assert all(
    oof["row_index"].nunique() == 513
    for oof in rolling_oof_predictions.values()
)


print(
    "\n✓ Every probability uses historical fixtures only."
)

print(
    "✓ 2025+ final test remains untouched."
)

ROLLING APPEARANCE-RATE BASELINES
 window mean_probability    brier log_loss roc_auc
    100           0.8292 0.138294 0.450864  0.5139
     50           0.8333 0.139354 0.455668  0.5137
     30           0.8367 0.140266 0.460390  0.5242
     20           0.8367 0.145288 0.605426  0.5007
     10           0.8366 0.147173 1.303521  0.5444

Existing benchmarks
-------------------------------------------------------
Historical rate : Brier 0.137300 | Log loss 0.447232
Full CatBoost   : Brier 0.137897 | Log loss 0.451310
Logistic        : Brier 0.149884 | Log loss 0.498344
Previous state  : Brier 0.141945 | Log loss 0.460873

Best rolling window
-------------------------------------------------------
Window: 100
Brier: 0.138294

✓ Every probability uses historical fixtures only.
✓ 2025+ final test remains untouched.


In [50]:
# Fold-by-fold model comparison

comparison_rows = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[
        fold["train_idx"]
    ]

    val = dev_df.loc[
        fold["val_idx"]
    ]

    y_val = val[target].to_numpy()
    actual_rate = y_val.mean()


    # Reconstruct historical-rate baseline directly.
    historical_probability = train[
        target
    ].mean()

    historical_prob = np.full(
        len(val),
        historical_probability,
    )


    comparison_rows.append({
        "fold": fold_number,
        "actual_rate": actual_rate,
        "model": "Historical rate",
        "predicted_rate": historical_prob.mean(),
        "rate_error": historical_prob.mean() - actual_rate,
        "brier": brier_score_loss(
            y_val,
            historical_prob,
        ),
        "log_loss": log_loss(
            y_val,
            historical_prob,
            labels=[0, 1],
        ),
    })


    # Previously evaluated models.
    model_results = {
        "Logistic": logistic_results,
        "CatBoost": catboost_results,
        "Previous state": state_results,
    }


    for model_name, results_df in model_results.items():

        row = results_df.loc[
            results_df["fold"].eq(fold_number)
        ].iloc[0]

        comparison_rows.append({
            "fold": fold_number,
            "actual_rate": actual_rate,
            "model": model_name,
            "predicted_rate": row["predicted_rate"],
            "rate_error": (
                row["predicted_rate"]
                - actual_rate
            ),
            "brier": row["brier"],
            "log_loss": row["log_loss"],
        })


comparison = pd.DataFrame(
    comparison_rows
)


print("MODEL PERFORMANCE BY TEMPORAL FOLD")
print("=" * 105)


for fold_number in sorted(
    comparison["fold"].unique()
):

    fold_table = comparison[
        comparison["fold"].eq(fold_number)
    ].copy()

    fold_table = fold_table.sort_values(
        "brier"
    )

    actual_rate = fold_table[
        "actual_rate"
    ].iloc[0]


    print(
        f"\nFOLD {fold_number} "
        f"— actual appearance rate: "
        f"{actual_rate:.4f}"
    )

    print("-" * 90)

    print(
        fold_table[
            [
                "model",
                "predicted_rate",
                "rate_error",
                "brier",
                "log_loss",
            ]
        ].to_string(
            index=False,
            formatters={
                "predicted_rate": "{:.4f}".format,
                "rate_error": "{:+.4f}".format,
                "brier": "{:.5f}".format,
                "log_loss": "{:.5f}".format,
            }
        )
    )


print("\nFold winners by Brier")
print("-" * 60)

winners = (
    comparison.loc[
        comparison
        .groupby("fold")["brier"]
        .idxmin(),
        [
            "fold",
            "model",
            "brier",
        ],
    ]
    .sort_values("fold")
)


print(
    winners.to_string(
        index=False,
        formatters={
            "brier": "{:.5f}".format
        }
    )
)


print("\nMean absolute appearance-rate error")
print("-" * 60)

rate_error_summary = (
    comparison
    .assign(
        abs_rate_error=lambda x:
            x["rate_error"].abs()
    )
    .groupby("model")[
        "abs_rate_error"
    ]
    .mean()
    .sort_values()
)


print(
    rate_error_summary.to_string(
        float_format=lambda x: f"{x:.4f}"
    )
)


assert len(comparison) == 16
assert comparison["fold"].nunique() == 4

print(
    "\n✓ Historical baseline reconstructed from each fold's training history."
)

print(
    "✓ Comparison uses temporal validation only."
)

print(
    "✓ Final 2025+ test remains untouched."
)

MODEL PERFORMANCE BY TEMPORAL FOLD

FOLD 1 — actual appearance rate: 0.7793
------------------------------------------------------------------------------------------
          model predicted_rate rate_error   brier log_loss
       CatBoost         0.8293    +0.0500 0.16449  0.52606
Historical rate         0.8498    +0.0705 0.17695  0.54521
       Logistic         0.8192    +0.0399 0.18133  0.57700
 Previous state         0.8328    +0.0535 0.19042  0.59000

FOLD 2 — actual appearance rate: 0.8261
------------------------------------------------------------------------------------------
          model predicted_rate rate_error   brier log_loss
 Previous state         0.8374    +0.0113 0.14120  0.45461
Historical rate         0.8406    +0.0146 0.14388  0.46281
       CatBoost         0.8537    +0.0276 0.16263  0.51901
       Logistic         0.8954    +0.0693 0.17910  0.63348

FOLD 3 — actual appearance rate: 0.8538
----------------------------------------------------------------------

In [52]:
# Shrink CatBoost probabilities toward historical-rate baseline

blend_weights = np.arange(
    0.0,
    1.01,
    0.10,
)

blend_parts = []


for fold in temporal_folds:

    fold_number = fold["fold"]

    train = dev_df.loc[
        fold["train_idx"]
    ]

    val = dev_df.loc[
        fold["val_idx"]
    ]


    # Historical probability learned from this fold's training data only.
    historical_probability = train[
        target
    ].mean()


    # CatBoost OOF predictions from Step 11.16.
    fold_catboost = catboost_oof.loc[
        catboost_oof["fold"].eq(fold_number)
    ].copy()


    fold_catboost = (
        fold_catboost
        .set_index("row_index")
        .loc[val.index]
        .reset_index()
    )


    blend_parts.append(
        pd.DataFrame({
            "row_index": val.index,
            "fold": fold_number,
            "actual": val[target].to_numpy(),

            "historical_probability":
                historical_probability,

            "catboost_probability":
                fold_catboost[
                    "predicted_probability"
                ].to_numpy(),
        })
    )


blend_oof = (
    pd.concat(
        blend_parts,
        ignore_index=True,
    )
    .sort_values("row_index")
    .reset_index(drop=True)
)


blend_results = []


for weight in blend_weights:

    probability = (
        (1 - weight)
        * blend_oof["historical_probability"]
        +
        weight
        * blend_oof["catboost_probability"]
    )


    blend_results.append({
        "catboost_weight": weight,

        "historical_weight":
            1 - weight,

        "mean_probability":
            probability.mean(),

        "brier":
            brier_score_loss(
                blend_oof["actual"],
                probability,
            ),

        "log_loss":
            log_loss(
                blend_oof["actual"],
                probability,
                labels=[0, 1],
            ),

        "roc_auc":
            roc_auc_score(
                blend_oof["actual"],
                probability,
            ),
    })


blend_results = (
    pd.DataFrame(
        blend_results
    )
    .sort_values("brier")
    .reset_index(drop=True)
)


print("CATBOOST + HISTORICAL-RATE SHRINKAGE")
print("=" * 90)

print(
    blend_results.to_string(
        index=False,
        formatters={
            "catboost_weight":
                "{:.1f}".format,

            "historical_weight":
                "{:.1f}".format,

            "mean_probability":
                "{:.4f}".format,

            "brier":
                "{:.6f}".format,

            "log_loss":
                "{:.6f}".format,

            "roc_auc":
                "{:.4f}".format,
        }
    )
)


best_row = blend_results.iloc[0]


print("\nBest OOF blend")
print("-" * 55)

print(
    "CatBoost weight:",
    f"{best_row['catboost_weight']:.1f}",
)

print(
    "Historical weight:",
    f"{best_row['historical_weight']:.1f}",
)

print(
    "Brier:",
    f"{best_row['brier']:.6f}",
)

print(
    "Log loss:",
    f"{best_row['log_loss']:.6f}",
)

print(
    "ROC AUC:",
    f"{best_row['roc_auc']:.4f}",
)


print("\nReference")
print("-" * 55)

print(
    "Historical only : "
    "Brier 0.137300 | "
    "Log loss 0.447232"
)

print(
    "CatBoost only   : "
    "Brier 0.137897 | "
    "Log loss 0.451310"
)


assert len(blend_oof) == 513
assert blend_oof["row_index"].nunique() == 513

assert blend_oof[
    [
        "historical_probability",
        "catboost_probability",
    ]
].notna().all().all()


print(
    "\n✓ Blend selection uses OOF development predictions only."
)

print(
    "✓ Final 2025+ test remains untouched."
)

CATBOOST + HISTORICAL-RATE SHRINKAGE
catboost_weight historical_weight mean_probability    brier log_loss roc_auc
            0.5               0.5           0.8375 0.134174 0.437879  0.5730
            0.4               0.6           0.8386 0.134251 0.437791  0.5722
            0.6               0.4           0.8365 0.134370 0.438790  0.5748
            0.3               0.7           0.8397 0.134602 0.438558  0.5716
            0.7               0.3           0.8354 0.134841 0.440528  0.5755
            0.2               0.8           0.8407 0.135227 0.440262  0.5709
            0.8               0.2           0.8343 0.135585 0.443131  0.5757
            0.1               0.9           0.8418 0.136127 0.443056  0.5604
            0.9               0.1           0.8332 0.136604 0.446680  0.5764
            0.0               1.0           0.8429 0.137300 0.447232  0.4578
            1.0               0.0           0.8322 0.137897 0.451310  0.5769

Best OOF blend
-----------------------

In [54]:
# Fold-by-fold validation of the selected 50/50 blend

SELECTED_CATBOOST_WEIGHT = 0.50
SELECTED_HISTORICAL_WEIGHT = 0.50

blend_fold_rows = []


for fold_number in sorted(
    blend_oof["fold"].unique()
):

    fold_data = blend_oof.loc[
        blend_oof["fold"].eq(fold_number)
    ].copy()


    y_true = fold_data[
        "actual"
    ].to_numpy()


    historical_prob = fold_data[
        "historical_probability"
    ].to_numpy()


    catboost_prob = fold_data[
        "catboost_probability"
    ].to_numpy()


    blend_prob = (
        SELECTED_HISTORICAL_WEIGHT
        * historical_prob
        +
        SELECTED_CATBOOST_WEIGHT
        * catboost_prob
    )


    blend_fold_rows.append({
        "fold": fold_number,

        "actual_rate":
            y_true.mean(),

        "historical_rate":
            historical_prob.mean(),

        "catboost_rate":
            catboost_prob.mean(),

        "blend_rate":
            blend_prob.mean(),

        "historical_brier":
            brier_score_loss(
                y_true,
                historical_prob,
            ),

        "catboost_brier":
            brier_score_loss(
                y_true,
                catboost_prob,
            ),

        "blend_brier":
            brier_score_loss(
                y_true,
                blend_prob,
            ),

        "historical_logloss":
            log_loss(
                y_true,
                historical_prob,
                labels=[0, 1],
            ),

        "catboost_logloss":
            log_loss(
                y_true,
                catboost_prob,
                labels=[0, 1],
            ),

        "blend_logloss":
            log_loss(
                y_true,
                blend_prob,
                labels=[0, 1],
            ),
    })


blend_fold_results = pd.DataFrame(
    blend_fold_rows
)


print("SELECTED 50/50 BLEND — TEMPORAL FOLD CHECK")
print("=" * 125)

print(
    blend_fold_results.to_string(
        index=False,
        formatters={
            "actual_rate": "{:.4f}".format,
            "historical_rate": "{:.4f}".format,
            "catboost_rate": "{:.4f}".format,
            "blend_rate": "{:.4f}".format,

            "historical_brier": "{:.5f}".format,
            "catboost_brier": "{:.5f}".format,
            "blend_brier": "{:.5f}".format,

            "historical_logloss": "{:.5f}".format,
            "catboost_logloss": "{:.5f}".format,
            "blend_logloss": "{:.5f}".format,
        }
    )
)


print("\nBrier winner by fold")
print("-" * 70)


for _, row in blend_fold_results.iterrows():

    scores = {
        "Historical": row["historical_brier"],
        "CatBoost": row["catboost_brier"],
        "50/50 Blend": row["blend_brier"],
    }

    winner = min(
        scores,
        key=scores.get,
    )

    print(
        f"Fold {int(row['fold'])}: "
        f"{winner} "
        f"({scores[winner]:.5f})"
    )


print("\nBlend improvement vs historical baseline")
print("-" * 70)


blend_fold_results[
    "brier_improvement_vs_historical"
] = (
    blend_fold_results[
        "historical_brier"
    ]
    -
    blend_fold_results[
        "blend_brier"
    ]
)


for _, row in blend_fold_results.iterrows():

    improvement = row[
        "brier_improvement_vs_historical"
    ]

    print(
        f"Fold {int(row['fold'])}: "
        f"{improvement:+.5f}"
    )


print("\nMacro averages")
print("-" * 70)

print(
    "Historical Brier:",
    f"{blend_fold_results['historical_brier'].mean():.6f}",
)

print(
    "CatBoost Brier:",
    f"{blend_fold_results['catboost_brier'].mean():.6f}",
)

print(
    "50/50 Blend Brier:",
    f"{blend_fold_results['blend_brier'].mean():.6f}",
)


assert len(blend_fold_results) == 4

assert np.allclose(
    SELECTED_CATBOOST_WEIGHT
    + SELECTED_HISTORICAL_WEIGHT,
    1.0,
)


print(
    "\n✓ Selected blend checked independently across all four temporal folds."
)

print(
    "✓ 2025+ final test is still untouched."
)

SELECTED 50/50 BLEND — TEMPORAL FOLD CHECK
 fold actual_rate historical_rate catboost_rate blend_rate historical_brier catboost_brier blend_brier historical_logloss catboost_logloss blend_logloss
    1      0.7793          0.8498        0.8293     0.8395          0.17695        0.16449     0.16683            0.54521          0.52606       0.51871
    2      0.8261          0.8406        0.8537     0.8472          0.14388        0.16263     0.14790            0.46281          0.51901       0.47381
    3      0.8538          0.8393        0.8387     0.8390          0.12500        0.12023     0.12040            0.41679          0.40197       0.40258
    4      0.8943          0.8407        0.8085     0.8246          0.09740        0.10209     0.09739            0.34934          0.35205       0.34631

Brier winner by fold
----------------------------------------------------------------------
Fold 1: CatBoost (0.16449)
Fold 2: Historical (0.14388)
Fold 3: CatBoost (0.12023)
Fold 4: 50/50 Bl

In [56]:
# One-time evaluation on the untouched 2025+ final test

FINAL_CATBOOST_WEIGHT = 0.50
FINAL_HISTORICAL_WEIGHT = 0.50


X_dev = dev_df[
    appearance_features
].copy()

y_dev = dev_df[
    target
].copy()


X_test = test_df[
    appearance_features
].copy()

y_test = test_df[
    target
].copy()


for col in categorical_features:
    X_dev[col] = X_dev[col].astype(str)
    X_test[col] = X_test[col].astype(str)


final_appearance_model = CatBoostClassifier(
    loss_function="Logloss",
    iterations=250,
    depth=3,
    learning_rate=0.03,
    l2_leaf_reg=10,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)


final_appearance_model.fit(
    X_dev,
    y_dev,
    cat_features=categorical_features,
)


catboost_test_prob = (
    final_appearance_model
    .predict_proba(X_test)[:, 1]
)


historical_probability = y_dev.mean()

historical_test_prob = np.full(
    len(test_df),
    historical_probability,
)


blend_test_prob = (
    FINAL_HISTORICAL_WEIGHT
    * historical_test_prob
    +
    FINAL_CATBOOST_WEIGHT
    * catboost_test_prob
)


def evaluate_probability_model(
    name,
    y_true,
    probability,
):

    return {
        "model": name,

        "actual_rate":
            y_true.mean(),

        "predicted_rate":
            probability.mean(),

        "rate_error":
            probability.mean()
            - y_true.mean(),

        "brier":
            brier_score_loss(
                y_true,
                probability,
            ),

        "log_loss":
            log_loss(
                y_true,
                probability,
                labels=[0, 1],
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability,
            ),
    }


final_test_results = pd.DataFrame([
    evaluate_probability_model(
        "Historical rate",
        y_test,
        historical_test_prob,
    ),

    evaluate_probability_model(
        "CatBoost",
        y_test,
        catboost_test_prob,
    ),

    evaluate_probability_model(
        "50/50 Blend",
        y_test,
        blend_test_prob,
    ),
])


print("APPEARANCE MODEL — FINAL 2025+ TEST")
print("=" * 95)

print(
    final_test_results.to_string(
        index=False,
        formatters={
            "actual_rate":
                "{:.4f}".format,

            "predicted_rate":
                "{:.4f}".format,

            "rate_error":
                "{:+.4f}".format,

            "brier":
                "{:.6f}".format,

            "log_loss":
                "{:.6f}".format,

            "roc_auc":
                "{:.4f}".format,
        }
    )
)


print("\nTest-set details")
print("-" * 60)

print(
    "Rows:",
    len(test_df),
)

print(
    "Date range:",
    test_df["date"].min().date(),
    "→",
    test_df["date"].max().date(),
)

print(
    "Appearances:",
    int(y_test.sum()),
)

print(
    "Nonappearances:",
    int((1 - y_test).sum()),
)

print(
    "Actual appearance rate:",
    f"{y_test.mean():.4f}",
)

print(
    "Development historical rate:",
    f"{historical_probability:.4f}",
)


print("\nSelected model")
print("-" * 60)

selected = final_test_results.loc[
    final_test_results["model"]
    .eq("50/50 Blend")
].iloc[0]

print(
    "Brier:",
    f"{selected['brier']:.6f}",
)

print(
    "Log loss:",
    f"{selected['log_loss']:.6f}",
)

print(
    "ROC AUC:",
    f"{selected['roc_auc']:.4f}",
)

print(
    "Mean predicted probability:",
    f"{selected['predicted_rate']:.4f}",
)


assert len(test_df) == 99
assert int(y_test.sum()) == 78
assert int((1 - y_test).sum()) == 21

assert np.all(
    (blend_test_prob >= 0)
    &
    (blend_test_prob <= 1)
)


print(
    "\n✓ Final specification was selected before opening the test."
)

print(
    "✓ 2025+ test evaluated once with no further tuning."
)

APPEARANCE MODEL — FINAL 2025+ TEST
          model actual_rate predicted_rate rate_error    brier log_loss roc_auc
Historical rate      0.7879         0.8451    +0.0572 0.170402 0.528209  0.5000
       CatBoost      0.7879         0.8485    +0.0606 0.156882 0.486236  0.6941
    50/50 Blend      0.7879         0.8468    +0.0589 0.161531 0.498748  0.6941

Test-set details
------------------------------------------------------------
Rows: 99
Date range: 2025-01-09 → 2026-08-28
Appearances: 78
Nonappearances: 21
Actual appearance rate: 0.7879
Development historical rate: 0.8451

Selected model
------------------------------------------------------------
Brier: 0.161531
Log loss: 0.498748
ROC AUC: 0.6941
Mean predicted probability: 0.8468

✓ Final specification was selected before opening the test.
✓ 2025+ test evaluated once with no further tuning.


In [58]:
# Final-test calibration diagnostic

final_test_predictions = test_df[
    [
        "date",
        "team",
        "opponent",
        "competition",
        "venue",
        "appeared",
    ]
].copy()


final_test_predictions[
    "catboost_probability"
] = catboost_test_prob

final_test_predictions[
    "historical_probability"
] = historical_test_prob

final_test_predictions[
    "appearance_probability"
] = blend_test_prob


# Use quantiles so the small 99-row test set is split reasonably evenly.
final_test_predictions[
    "probability_bin"
] = pd.qcut(
    final_test_predictions[
        "appearance_probability"
    ],
    q=5,
    duplicates="drop",
)


calibration_table = (
    final_test_predictions
    .groupby(
        "probability_bin",
        observed=True,
    )
    .agg(
        fixtures=("appeared", "size"),
        appearances=("appeared", "sum"),
        predicted_probability=(
            "appearance_probability",
            "mean",
        ),
        actual_rate=("appeared", "mean"),
        min_probability=(
            "appearance_probability",
            "min",
        ),
        max_probability=(
            "appearance_probability",
            "max",
        ),
    )
    .reset_index()
)


calibration_table[
    "calibration_error"
] = (
    calibration_table[
        "predicted_probability"
    ]
    -
    calibration_table[
        "actual_rate"
    ]
)


print("FINAL TEST — 50/50 BLEND CALIBRATION")
print("=" * 105)

print(
    calibration_table.to_string(
        index=False,
        formatters={
            "predicted_probability":
                "{:.4f}".format,

            "actual_rate":
                "{:.4f}".format,

            "min_probability":
                "{:.4f}".format,

            "max_probability":
                "{:.4f}".format,

            "calibration_error":
                "{:+.4f}".format,
        }
    )
)


print("\nOverall")
print("-" * 60)

print(
    "Fixtures:",
    len(final_test_predictions),
)

print(
    "Predicted appearance rate:",
    f"{blend_test_prob.mean():.4f}",
)

print(
    "Actual appearance rate:",
    f"{y_test.mean():.4f}",
)

print(
    "Overall calibration error:",
    f"{blend_test_prob.mean() - y_test.mean():+.4f}",
)


# Ranking of predicted probabilities by actual outcome.
print("\nProbability by actual outcome")
print("-" * 60)

outcome_summary = (
    final_test_predictions
    .groupby("appeared")[
        "appearance_probability"
    ]
    .agg(
        fixtures="size",
        mean_probability="mean",
        median_probability="median",
    )
    .reset_index()
)

print(
    outcome_summary.to_string(
        index=False,
        formatters={
            "mean_probability": "{:.4f}".format,
            "median_probability": "{:.4f}".format,
        }
    )
)


assert len(final_test_predictions) == 99
assert final_test_predictions[
    "appearance_probability"
].between(0, 1).all()


print(
    "\n✓ Final test used for diagnosis only — no model selection or retuning."
)

print(
    "✓ Locked 50/50 appearance model remains unchanged."
)

FINAL TEST — 50/50 BLEND CALIBRATION
probability_bin  fixtures  appearances predicted_probability actual_rate min_probability max_probability calibration_error
 (0.617, 0.813]        20           15                0.7766      0.7500          0.6184          0.8130           +0.0266
 (0.813, 0.845]        20           10                0.8252      0.5000          0.8138          0.8442           +0.3252
 (0.845, 0.877]        19           18                0.8667      0.9474          0.8470          0.8763           -0.0806
 (0.877, 0.883]        20           16                0.8801      0.8000          0.8770          0.8832           +0.0801
 (0.883, 0.889]        20           19                0.8864      0.9500          0.8835          0.8890           -0.0636

Overall
------------------------------------------------------------
Fixtures: 99
Predicted appearance rate: 0.8468
Actual appearance rate: 0.7879
Overall calibration error: +0.0589

Probability by actual outcome
-----------

In [60]:
# Save locked appearance-model specification and predictions

import json
from pathlib import Path


ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


appearance_model_spec = {
    "model_name": "catboost_historical_rate_blend",

    "target": "appeared",

    "training_cutoff": "2024-12-31",
    "final_test_start": "2025-01-01",
    "historical_data_cutoff": "2026-08-28",

    "features": appearance_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,

    "catboost": {
        "loss_function": "Logloss",
        "iterations": 250,
        "depth": 3,
        "learning_rate": 0.03,
        "l2_leaf_reg": 10,
        "random_seed": 42,
    },

    "blend": {
        "catboost_weight": 0.50,
        "historical_rate_weight": 0.50,
    },

    "selection_metric": "Brier score",

    "development_oof": {
        "rows": 513,
        "brier": 0.134174,
        "log_loss": 0.437879,
        "roc_auc": 0.5730,
    },

    "final_test": {
        "rows": 99,
        "appearances": 78,
        "nonappearances": 21,
        "actual_rate": 0.7879,
        "predicted_rate": 0.8468,
        "brier": 0.161531,
        "log_loss": 0.498748,
        "roc_auc": 0.6941,
    },

    "calibration_note": (
        "The locked model overpredicted the overall appearance rate "
        "during the 2025-2026 final test period. The final test was "
        "used for evaluation only and no post-test recalibration or "
        "model selection was performed."
    ),
}


spec_path = (
    ARTIFACT_DIR
    / "appearance_model_spec.json"
)

prediction_path = (
    ARTIFACT_DIR
    / "appearance_final_test_predictions.csv"
)


with open(
    spec_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        appearance_model_spec,
        f,
        indent=2,
    )


final_test_predictions[
    [
        "date",
        "team",
        "opponent",
        "competition",
        "venue",
        "appeared",
        "historical_probability",
        "catboost_probability",
        "appearance_probability",
    ]
].to_csv(
    prediction_path,
    index=False,
)


print("APPEARANCE MODEL ARTIFACTS SAVED")
print("=" * 70)

print(
    "Model specification:",
    spec_path,
)

print(
    "Final test predictions:",
    prediction_path,
)


print("\nLocked model")
print("-" * 50)

print(
    "50% CatBoost + 50% historical appearance rate"
)

print(
    "OOF Brier:       0.134174"
)

print(
    "Final-test Brier: 0.161531"
)

print(
    "Final-test AUC:   0.6941"
)


assert spec_path.exists()
assert prediction_path.exists()


print(
    "\n✓ Appearance model specification locked and saved."
)

print(
    "✓ Final-test predictions preserved."
)

print(
    "✓ Notebook 11 appearance modeling is complete."
)

APPEARANCE MODEL ARTIFACTS SAVED
Model specification: ..\artifacts\appearance_model_spec.json
Final test predictions: ..\artifacts\appearance_final_test_predictions.csv

Locked model
--------------------------------------------------
50% CatBoost + 50% historical appearance rate
OOF Brier:       0.134174
Final-test Brier: 0.161531
Final-test AUC:   0.6941

✓ Appearance model specification locked and saved.
✓ Final-test predictions preserved.
✓ Notebook 11 appearance modeling is complete.
